# grad-tracking-global-toggle — worked example 1: NoGrad context manager via module-level toggle

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `grad-tracking-global-toggle`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

A module-level boolean flag controls whether the autograd graph is built during a forward pass. The `NoGrad` context manager flips this flag to `False` on entry and restores the PREVIOUS value on exit — not a hard-coded `True`. This nesting-safe pattern means that if `NoGrad` is used inside another `NoGrad`, the inner exit does not accidentally re-enable grad tracking while the outer is still active.

## Worked solution

**Step 1 — define the module-level flag.** `grad_tracking_enabled = True` lives at module (global) scope. All functions that check it must read the global dynamically, not snapshot it at import time.

**Step 2 — implement compute_requires_grad.** Returns `True` only when the flag is `True` AND at least one argument has `requires_grad=True`. Both conditions together prevent useless graph construction during inference.

**Step 3 — implement NoGrad.__enter__.** Save the CURRENT flag value, then set it to `False`. Using `self._prev` rather than assuming the previous state was `True` is what makes nesting work.

**Step 4 — implement NoGrad.__exit__.** Restore `self._prev`. Return `False` so any exception from the body propagates normally.

**Step 5 — demonstrate nested use.** Two nested `NoGrad` blocks: the inner exit restores `False` (not `True`), so the outer block is still correctly disabled.

In [ ]:
import torch as t

grad_tracking_enabled = True

def _get_flag():
    return globals()['grad_tracking_enabled']

def _set_flag(v):
    globals()['grad_tracking_enabled'] = v

class MiniTensor:
    def __init__(self, data, requires_grad=False):
        self.data = data
        self.requires_grad = requires_grad

def compute_requires_grad(args):
    return _get_flag() and any(
        isinstance(a, MiniTensor) and a.requires_grad for a in args
    )

class NoGrad:
    def __enter__(self):
        self._prev = _get_flag()
        _set_flag(False)
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        _set_flag(self._prev)
        return False

# Exercise: outside any context, a leaf tensor requires grad
x = MiniTensor(t.randn(3), requires_grad=True)
print(f"Outside NoGrad: {compute_requires_grad([x])}")  # True

with NoGrad():
    print(f"Inside NoGrad:  {compute_requires_grad([x])}")  # False
    with NoGrad():
        print(f"  Nested NoGrad: {compute_requires_grad([x])}")  # False
    print(f"After inner exit (still in outer): {compute_requires_grad([x])}")  # False

print(f"After outer exit: {compute_requires_grad([x])}")  # True